# CIS 3400 – Advanced Programming: Walmart Stock & Freight Tracking System

**Course:** CIS 3400 – Advanced Programming  
**Project:** Project 2 – Python Application Development  
**System Overview:** End-to-end store inventory and freight lifecycle tracker monitoring incoming trailers, backroom staging bins, sales floor shelf replenishment, and a full 30-day store simulation with strict foreign-key SKU integrity and zero-negative inventory guarantees.

---
### Notebook Architecture
1. **Setup & Dependencies** – Import standard data science and SQLite libraries.
2. **Database Schema (SQLite with Strict FK & Non-Negative Constraints)** – Enforce `PRAGMA foreign_keys = ON` and `CHECK (qty >= 0)` across `products`, `freight_shipments`, and `movement_logs`.
3. **Seed Product Catalog** – Populate 10 core store SKUs across 5 departments.
4. **Core Application Engine (`WalmartFreightStockTracker`)** – Object-oriented Python backend validating every SKU against `products` and guaranteeing non-negative floor and backroom balances.
5. **30-Day Store Operations Simulation (24+ Freight Shipments & 60+ Movement Logs)** – Simulates a full month of daily receiving, backroom-to-shelf replenishment, and POS customer sales.
6. **Visual Analytics Dashboard** – Charts inventory health, freight status breakdown, and 30-day stock movement trends.
7. **Automated Testing & Validation** – Audits foreign-key SKU matching, non-negative inventory invariants, record counts (>=20 freight & >=20 movement logs), and error handling.

## 1. Setup & Dependencies

In [ ]:
import sqlite3
import random
from datetime import datetime, timedelta
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Set random seed for reproducible 30-day simulation
random.seed(42)
np.random.seed(42)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

DB_PATH = 'walmart_store_tracker.db'
print('Environment initialized successfully.')

## 2. Database Schema & Initialization
We enforce strict relational and data integrity rules at the SQLite engine level:
- **`PRAGMA foreign_keys = ON`**: Every `sku` inserted into `freight_shipments` or `movement_logs` **must** match an existing `sku` in `products`.
- **`CHECK (floor_qty >= 0)` & `CHECK (backroom_qty >= 0)`**: Database-level constraints make negative inventory impossible.

In [ ]:
def get_db_connection(db_path=DB_PATH):
    conn = sqlite3.connect(db_path)
    conn.execute('PRAGMA foreign_keys = ON;')
    return conn

def initialize_database(db_path=DB_PATH):
    conn = get_db_connection(db_path)
    cursor = conn.cursor()

    cursor.executescript('''
    PRAGMA foreign_keys = OFF;
    DROP TABLE IF EXISTS movement_logs;
    DROP TABLE IF EXISTS freight_shipments;
    DROP TABLE IF EXISTS products;
    PRAGMA foreign_keys = ON;

    CREATE TABLE products (
        sku TEXT PRIMARY KEY,
        item_name TEXT NOT NULL,
        department TEXT NOT NULL,
        aisle_location TEXT NOT NULL,
        backroom_bin TEXT NOT NULL,
        floor_qty INTEGER NOT NULL DEFAULT 0 CHECK (floor_qty >= 0),
        backroom_qty INTEGER NOT NULL DEFAULT 0 CHECK (backroom_qty >= 0),
        reorder_point INTEGER NOT NULL DEFAULT 20 CHECK (reorder_point >= 0),
        avg_daily_sales REAL NOT NULL DEFAULT 5.0,
        unit_price REAL NOT NULL CHECK (unit_price >= 0)
    );

    CREATE TABLE freight_shipments (
        shipment_id TEXT PRIMARY KEY,
        trailer_number TEXT NOT NULL,
        dc_origin TEXT NOT NULL,
        sku TEXT NOT NULL,
        quantity INTEGER NOT NULL CHECK (quantity > 0),
        status TEXT NOT NULL CHECK (status IN ('In Transit', 'At Dock', 'Unloaded', 'Stocked')),
        eta TEXT NOT NULL,
        FOREIGN KEY (sku) REFERENCES products(sku) ON UPDATE CASCADE ON DELETE RESTRICT
    );

    CREATE TABLE movement_logs (
        log_id INTEGER PRIMARY KEY AUTOINCREMENT,
        timestamp TEXT NOT NULL,
        sku TEXT NOT NULL,
        action_type TEXT NOT NULL CHECK (action_type IN ('FREIGHT_UNLOAD', 'FLOOR_REPLENISH', 'POS_SALE')),
        quantity_change INTEGER NOT NULL,
        resulting_floor_qty INTEGER NOT NULL CHECK (resulting_floor_qty >= 0),
        resulting_backroom_qty INTEGER NOT NULL CHECK (resulting_backroom_qty >= 0),
        source_location TEXT NOT NULL,
        destination_location TEXT NOT NULL,
        notes TEXT,
        FOREIGN KEY (sku) REFERENCES products(sku) ON UPDATE CASCADE ON DELETE RESTRICT
    );
    ''')

    conn.commit()
    conn.close()
    print(f'Database schema initialized at {db_path} with strict FK & non-negative constraints.')

initialize_database()

## 3. Seed Master Product Catalog
All freight shipments and movement logs will strictly reference these 10 master product SKUs.

In [ ]:
def seed_product_catalog(db_path=DB_PATH):
    conn = get_db_connection(db_path)
    cursor = conn.cursor()

    sample_products = [
        ('WM-1001', 'Great Value Whole Milk 1 Gal', 'Grocery', 'Aisle G12', 'BR-COOL-01', 25, 50, 25, 16.0, 3.68),
        ('WM-1002', 'Great Value Purified Water 40pk', 'Grocery', 'Aisle G04', 'BR-DRY-04', 30, 70, 30, 20.0, 5.36),
        ('WM-1003', 'Great Value Large White Eggs 18ct', 'Grocery', 'Aisle G11', 'BR-COOL-02', 20, 45, 20, 14.0, 4.12),
        ('WM-2010', 'onn. 55" 4K UHD Roku TV', 'Electronics', 'Aisle E01', 'BR-SEC-02', 6, 14, 5, 2.0, 248.00),
        ('WM-2015', 'Apple AirPods Pro (2nd Gen)', 'Electronics', 'Aisle E05', 'BR-SEC-01', 10, 20, 8, 3.0, 189.00),
        ('WM-3040', 'Bounty Select-A-Size Paper Towels 12pk', 'Household', 'Aisle H02', 'BR-BULK-03', 18, 40, 15, 8.5, 22.44),
        ('WM-3045', 'Tide Pods Liquid Laundry Detergent 81ct', 'Household', 'Aisle H06', 'BR-BULK-01', 22, 35, 15, 6.5, 21.44),
        ('WM-4012', 'George Men Crewneck Tee 3pk', 'Apparel', 'Zone A03', 'BR-APP-02', 28, 45, 20, 5.5, 14.98),
        ('WM-4018', 'Athletic Works Running Socks 6pk', 'Apparel', 'Zone A07', 'BR-APP-04', 24, 40, 18, 5.0, 11.98),
        ('WM-5008', 'Equate Ibuprofen 200mg 500ct', 'Health & Wellness', 'Aisle P03', 'BR-PHM-01', 18, 30, 15, 4.0, 9.97)
    ]

    cursor.executemany('''
        INSERT INTO products VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    ''', sample_products)

    conn.commit()
    conn.close()
    print(f'Loaded {len(sample_products)} master SKUs into products table.')

seed_product_catalog()

## 4. Core Application Engine (`WalmartFreightStockTracker`)
Every operation validates:
1. **SKU Existence**: Ensures `sku` exists in `products` before scheduling freight, unloading, restocking, or recording sales.
2. **Non-Negative Stock Guard**: Prevents `floor_qty` or `backroom_qty` from ever dropping below `0`.
3. **Synchronized Audit Logging**: Every movement records post-transaction `resulting_floor_qty` and `resulting_backroom_qty`.

In [ ]:
class WalmartFreightStockTracker:
    def __init__(self, db_path=DB_PATH):
        self.db_path = db_path

    def _connect(self):
        return get_db_connection(self.db_path)

    def _validate_sku(self, cursor, sku: str):
        cursor.execute('''
            SELECT sku, item_name, aisle_location, backroom_bin, floor_qty, backroom_qty, reorder_point
            FROM products WHERE sku = ?
        ''', (sku,))
        row = cursor.fetchone()
        if not row:
            raise ValueError(f'SKU {sku} does not exist in the master products table.')
        return row

    def _log_movement(self, cursor, sku, action_type, qty_change, floor_after, backroom_after, src, dst, notes='', timestamp=None):
        if floor_after < 0 or backroom_after < 0:
            raise ValueError(f'Negative inventory prevented for {sku}: floor={floor_after}, backroom={backroom_after}')
        ts = timestamp or datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        cursor.execute('''
            INSERT INTO movement_logs (
                timestamp, sku, action_type, quantity_change,
                resulting_floor_qty, resulting_backroom_qty,
                source_location, destination_location, notes
            )
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
        ''', (ts, sku, action_type, qty_change, floor_after, backroom_after, src, dst, notes))

    def schedule_freight(self, shipment_id: str, trailer_number: str, dc_origin: str, sku: str, quantity: int, status: str, eta: str) -> str:
        """Creates a freight shipment strictly linked to a valid product SKU."""
        if quantity <= 0:
            raise ValueError('Freight quantity must be greater than zero.')
        conn = self._connect()
        cursor = conn.cursor()
        try:
            self._validate_sku(cursor, sku)
            cursor.execute('''
                INSERT INTO freight_shipments (shipment_id, trailer_number, dc_origin, sku, quantity, status, eta)
                VALUES (?, ?, ?, ?, ?, ?, ?)
            ''', (shipment_id, trailer_number, dc_origin, sku, quantity, status, eta))
            conn.commit()
            return f'[SCHEDULED] {shipment_id} ({quantity}x {sku}) on {trailer_number} [{status}]'
        finally:
            conn.close()

    def receive_freight(self, shipment_id: str, timestamp: str = None) -> str:
        """Unloads a freight shipment from the dock into backroom bin inventory."""
        conn = self._connect()
        cursor = conn.cursor()
        try:
            cursor.execute('SELECT sku, quantity, status, trailer_number FROM freight_shipments WHERE shipment_id = ?', (shipment_id,))
            row = cursor.fetchone()
            if not row:
                raise ValueError(f'Shipment ID {shipment_id} not found.')

            sku, qty, status, trailer = row
            if status in ('Unloaded', 'Stocked'):
                raise ValueError(f'Shipment {shipment_id} has already been processed (Status: {status}).')

            _, _, _, bin_loc, floor_qty, backroom_qty, _ = self._validate_sku(cursor, sku)
            new_backroom = backroom_qty + qty

            cursor.execute('UPDATE products SET backroom_qty = ? WHERE sku = ?', (new_backroom, sku))
            cursor.execute("UPDATE freight_shipments SET status = 'Unloaded' WHERE shipment_id = ?", (shipment_id,))
            self._log_movement(
                cursor, sku, 'FREIGHT_UNLOAD', qty, floor_qty, new_backroom,
                f'Trailer {trailer}', bin_loc, f'Received {shipment_id}', timestamp=timestamp
            )

            conn.commit()
            return f'[UNLOADED] {shipment_id}: +{qty} units of {sku} into {bin_loc} (Backroom: {new_backroom})'
        finally:
            conn.close()

    def stock_sales_floor(self, sku: str, quantity: int, timestamp: str = None) -> str:
        """Transfers inventory from backroom bin to sales floor aisle without ever allowing negative backroom stock."""
        if quantity <= 0:
            raise ValueError('Transfer quantity must be greater than zero.')

        conn = self._connect()
        cursor = conn.cursor()
        try:
            _, _, aisle_loc, bin_loc, floor_qty, backroom_qty, _ = self._validate_sku(cursor, sku)
            if backroom_qty < quantity:
                raise ValueError(f'Insufficient backroom stock for {sku}. Requested: {quantity}, Available: {backroom_qty}.')

            new_backroom = backroom_qty - quantity
            new_floor = floor_qty + quantity

            cursor.execute('''
                UPDATE products SET backroom_qty = ?, floor_qty = ? WHERE sku = ?
            ''', (new_backroom, new_floor, sku))
            self._log_movement(
                cursor, sku, 'FLOOR_REPLENISH', quantity, new_floor, new_backroom,
                bin_loc, aisle_loc, 'Backroom-to-shelf replenishment', timestamp=timestamp
            )

            conn.commit()
            return f'[RESTOCKED] {sku}: moved {quantity} units {bin_loc} -> {aisle_loc} (Floor: {new_floor}, Backroom: {new_backroom})'
        finally:
            conn.close()

    def record_pos_sale(self, sku: str, quantity: int, timestamp: str = None) -> str:
        """Deducts sold units from sales floor inventory without ever allowing negative floor stock."""
        if quantity <= 0:
            raise ValueError('Sale quantity must be positive.')

        conn = self._connect()
        cursor = conn.cursor()
        try:
            _, _, aisle_loc, _, floor_qty, backroom_qty, reorder_pt = self._validate_sku(cursor, sku)
            if floor_qty < quantity:
                raise ValueError(f'Insufficient sales floor stock for {sku}. Floor Qty: {floor_qty}, Requested Sale: {quantity}.')

            new_floor = floor_qty - quantity
            cursor.execute('UPDATE products SET floor_qty = ? WHERE sku = ?', (new_floor, sku))
            self._log_movement(
                cursor, sku, 'POS_SALE', -quantity, new_floor, backroom_qty,
                aisle_loc, 'Customer POS Checkout', 'Customer purchase', timestamp=timestamp
            )

            conn.commit()
            alert = f' [LOW FLOOR ALERT < {reorder_pt}]' if new_floor < reorder_pt else ''
            return f'[POS SALE] {sku}: sold {quantity} units (Floor: {new_floor}).{alert}'
        finally:
            conn.close()

    def get_inventory_df(self) -> pd.DataFrame:
        conn = self._connect()
        df = pd.read_sql_query('SELECT * FROM products ORDER BY sku', conn)
        conn.close()
        df['total_on_hand'] = df['floor_qty'] + df['backroom_qty']
        df['days_of_supply'] = np.round(df['total_on_hand'] / df['avg_daily_sales'], 1)
        df['shelf_status'] = np.where(df['floor_qty'] < df['reorder_point'], 'NEEDS RESTOCK', 'OK')
        return df

    def get_freight_df(self) -> pd.DataFrame:
        conn = self._connect()
        df = pd.read_sql_query('''
            SELECT f.shipment_id, f.trailer_number, f.dc_origin, f.sku, p.item_name, f.quantity, f.status, f.eta
            FROM freight_shipments f
            JOIN products p ON f.sku = p.sku
            ORDER BY f.eta ASC, f.shipment_id ASC
        ''', conn)
        conn.close()
        return df

    def get_movement_logs(self, limit=None) -> pd.DataFrame:
        conn = self._connect()
        query = '''
            SELECT m.log_id, m.timestamp, m.sku, p.item_name, m.action_type,
                   m.quantity_change, m.resulting_floor_qty, m.resulting_backroom_qty,
                   m.source_location, m.destination_location, m.notes
            FROM movement_logs m
            JOIN products p ON m.sku = p.sku
            ORDER BY m.log_id ASC
        '''
        if limit:
            query += f' LIMIT {int(limit)}'
        df = pd.read_sql_query(query, conn)
        conn.close()
        return df

tracker = WalmartFreightStockTracker()
print('WalmartFreightStockTracker engine ready.')

## 5. Full 30-Day Store Operations Simulation
This simulation runs **30 consecutive days** of store operations:
- Schedules **24 freight shipments** (`FR-1001` through `FR-1024`) from Walmart Regional Distribution Centers using only valid SKUs from `products`.
- Unloads arriving shipments into backroom bins (leaving the final 4 shipments in `At Dock` and `In Transit` states to represent live active pipeline).
- Automatically replenishes sales floor shelves from backroom bins whenever floor stock dips below the SKU's `reorder_point` (capped strictly by available `backroom_qty`).
- Simulates daily customer POS sales capped strictly by available `floor_qty` so inventory **never goes negative**.
- Generates **24 freight shipment records** and **100+ movement logs** across the 30-day window.

In [ ]:
def run_monthly_simulation(tracker: WalmartFreightStockTracker, days: int = 30):
    start_date = datetime(2026, 11, 1, 6, 0, 0)
    dcs = ['DC #6040 (Shelby, NC)', 'DC #6011 (Brookhaven, MS)', 'DC #6072 (Pageland, SC)']
    inv_snapshot = tracker.get_inventory_df()
    skus = inv_snapshot['sku'].tolist()
    daily_rates = dict(zip(inv_snapshot['sku'], inv_snapshot['avg_daily_sales']))

    shipment_counter = 1001
    scheduled_shipments = []

    # Schedule 24 freight shipments spread across the 30-day month (8 delivery waves x 3 SKUs)
    delivery_days = [1, 4, 8, 12, 16, 20, 25, 29]
    for wave_idx, day_offset in enumerate(delivery_days):
        trailer_num = f'TRL-{4800 + wave_idx * 17}'
        dc = dcs[wave_idx % len(dcs)]
        eta_dt = start_date + timedelta(days=day_offset, hours=5)
        eta_str = eta_dt.strftime('%Y-%m-%d %H:%M')

        # Pick 3 distinct valid SKUs per trailer
        wave_skus = [skus[(wave_idx * 3 + j) % len(skus)] for j in range(3)]
        for sku in wave_skus:
            ship_id = f'FR-{shipment_counter}'
            shipment_counter += 1
            # Order roughly 6 days worth of average sales
            qty = max(12, int(round(daily_rates[sku] * 6)))
            initial_status = 'In Transit'
            tracker.schedule_freight(ship_id, trailer_num, dc, sku, qty, initial_status, eta_str)
            scheduled_shipments.append((day_offset, ship_id, eta_dt))

    # Simulate each of the 30 days
    for day in range(days):
        current_day_dt = start_date + timedelta(days=day)

        # 1. Morning Receiving (06:30 AM): Unload shipments scheduled for today (except last wave kept active for live view)
        for ship_day, ship_id, eta_dt in scheduled_shipments:
            if ship_day == day:
                if day < 29:
                    unload_ts = (current_day_dt + timedelta(hours=0, minutes=30)).strftime('%Y-%m-%d %H:%M:%S')
                    tracker.receive_freight(ship_id, timestamp=unload_ts)
                else:
                    # Mark final day trailer as 'At Dock' awaiting unload
                    conn = tracker._connect()
                    conn.execute("UPDATE freight_shipments SET status = 'At Dock' WHERE shipment_id = ?", (ship_id,))
                    conn.commit()
                    conn.close()

        # 2. Morning Shelf Replenishment (08:00 AM): Pull from backroom if floor < reorder_point
        current_inv = tracker.get_inventory_df()
        for _, row in current_inv.iterrows():
            if row['floor_qty'] < row['reorder_point'] and row['backroom_qty'] > 0:
                needed = (row['reorder_point'] * 2) - row['floor_qty']
                pull_qty = int(min(row['backroom_qty'], needed))
                if pull_qty > 0:
                    restock_ts = (current_day_dt + timedelta(hours=2)).strftime('%Y-%m-%d %H:%M:%S')
                    tracker.stock_sales_floor(row['sku'], pull_qty, timestamp=restock_ts)

        # 3. Daily POS Customer Sales (06:00 PM): Capped strictly by available floor_qty
        current_inv = tracker.get_inventory_df()
        for _, row in current_inv.iterrows():
            floor_avail = int(row['floor_qty'])
            if floor_avail > 0:
                # Simulate realistic daily demand around avg_daily_sales
                demand = max(1, int(np.random.poisson(row['avg_daily_sales'] * 0.65)))
                actual_sold = min(floor_avail, demand)
                if actual_sold > 0:
                    sale_ts = (current_day_dt + timedelta(hours=12)).strftime('%Y-%m-%d %H:%M:%S')
                    tracker.record_pos_sale(row['sku'], actual_sold, timestamp=sale_ts)

run_monthly_simulation(tracker, days=30)

freight_df = tracker.get_freight_df()
logs_df = tracker.get_movement_logs()
inv_df = tracker.get_inventory_df()

print(f'Simulation complete! Total Freight Shipments: {len(freight_df)} | Total Movement Logs: {len(logs_df)}')
print('\n=== FINAL STORE INVENTORY (END OF 30-DAY MONTH) ===')
display(inv_df)

print(f'\n=== FREIGHT SHIPMENTS TABLE ({len(freight_df)} RECORDS) ===')
display(freight_df)

print(f'\n=== MOVEMENT LOGS SAMPLE (FIRST 25 OF {len(logs_df)} RECORDS) ===')
display(logs_df.head(25))

## 6. Store Stock & 30-Day Freight Analytics Dashboard

In [ ]:
inv_df = tracker.get_inventory_df()
freight_df = tracker.get_freight_df()
logs_df = tracker.get_movement_logs()

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Chart 1: Sales Floor vs Backroom Stock by SKU
x = np.arange(len(inv_df))
width = 0.38
axes[0, 0].bar(x - width/2, inv_df['floor_qty'], width, label='Sales Floor Qty', color='#0071ce')
axes[0, 0].bar(x + width/2, inv_df['backroom_qty'], width, label='Backroom Bin Qty', color='#ffc220')
axes[0, 0].set_title('End-of-Month Sales Floor vs. Backroom Stock', fontsize=12, fontweight='bold')
axes[0, 0].set_xticks(x)
axes[0, 0].set_xticklabels(inv_df['sku'], rotation=45)
axes[0, 0].set_ylabel('Units')
axes[0, 0].legend()
axes[0, 0].grid(axis='y', linestyle='--', alpha=0.5)

# Chart 2: Days of Supply Remaining
colors = ['#d9534f' if d < 3.0 else '#28a745' for d in inv_df['days_of_supply']]
axes[0, 1].barh(inv_df['item_name'], inv_df['days_of_supply'], color=colors)
axes[0, 1].axvline(3.0, color='red', linestyle='--', linewidth=1, label='3-Day Safety Threshold')
axes[0, 1].set_title('Estimated Days of Supply Remaining', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Days of Supply')
axes[0, 1].legend()
axes[0, 1].grid(axis='x', linestyle='--', alpha=0.5)

# Chart 3: Freight Shipments by Status
status_counts = freight_df['status'].value_counts()
axes[1, 0].bar(status_counts.index, status_counts.values, color=['#28a745', '#ffc220', '#0071ce'])
axes[1, 0].set_title(f'Freight Shipments Breakdown (Total: {len(freight_df)})', fontsize=12, fontweight='bold')
axes[1, 0].set_ylabel('Number of Shipments')
axes[1, 0].grid(axis='y', linestyle='--', alpha=0.5)

# Chart 4: 30-Day Movement Activity by Action Type
action_counts = logs_df['action_type'].value_counts()
axes[1, 1].bar(action_counts.index, action_counts.values, color=['#d9534f', '#0071ce', '#28a745'])
axes[1, 1].set_title(f'30-Day Movement Logs by Type (Total: {len(logs_df)})', fontsize=12, fontweight='bold')
axes[1, 1].set_ylabel('Logged Transactions')
axes[1, 1].grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

## 7. Automated Data Integrity & Validation Audit
Verifies every requirement across the database:
1. **100% SKU Match**: Every SKU in `freight_shipments` and `movement_logs` exists in `products`.
2. **Zero Negative Inventory**: Confirms `floor_qty >= 0` and `backroom_qty >= 0` currently and across every historical log entry.
3. **Simulation Volume**: Confirms at least 20 freight shipments and at least 20 movement logs exist.
4. **Negative-Stock & Invalid-SKU Guards**: Confirms invalid operations raise clean `ValueError` exceptions.

In [ ]:
def run_validation_suite():
    results = []
    inv = tracker.get_inventory_df()
    freight = tracker.get_freight_df()
    logs = tracker.get_movement_logs()
    valid_skus = set(inv['sku'])

    # Check 1: Freight SKUs match Product SKUs
    freight_skus_valid = set(freight['sku']).issubset(valid_skus)
    assert freight_skus_valid, 'Found freight SKU not in products table!'
    results.append(('Freight-to-Product SKU Match', '100% of freight SKUs exist in products', f'PASSED ({len(freight)}/{len(freight)} matched)'))

    # Check 2: Movement Log SKUs match Product SKUs
    log_skus_valid = set(logs['sku']).issubset(valid_skus)
    assert log_skus_valid, 'Found movement log SKU not in products table!'
    results.append(('Movement-to-Product SKU Match', '100% of log SKUs exist in products', f'PASSED ({len(logs)}/{len(logs)} matched)'))

    # Check 3: Zero Negative Inventory (Current & Historical)
    min_floor_now = int(inv['floor_qty'].min())
    min_backroom_now = int(inv['backroom_qty'].min())
    min_floor_hist = int(logs['resulting_floor_qty'].min())
    min_backroom_hist = int(logs['resulting_backroom_qty'].min())
    assert min(min_floor_now, min_backroom_now, min_floor_hist, min_backroom_hist) >= 0, 'Negative inventory detected!'
    results.append(('Zero Negative Inventory', 'All floor & backroom qtys >= 0 across 30 days', f'PASSED (Min floor={min_floor_hist}, Min backroom={min_backroom_hist})'))

    # Check 4: Minimum 20 Freight Shipments & 20 Movement Logs
    assert len(freight) >= 20, f'Expected >= 20 freight shipments, got {len(freight)}'
    assert len(logs) >= 20, f'Expected >= 20 movement logs, got {len(logs)}'
    results.append(('30-Day Simulation Volume', '>= 20 freight shipments & >= 20 movement logs', f'PASSED ({len(freight)} freight, {len(logs)} logs)'))

    # Check 5: Foreign-key / Invalid SKU Rejection
    try:
        tracker.schedule_freight('FR-BAD', 'TRL-000', 'DC #0000', 'NON-EXISTENT-SKU', 10, 'In Transit', '2026-12-01 08:00')
        results.append(('Invalid SKU Guard', 'Reject unknown SKU', 'FAILED'))
    except ValueError as e:
        results.append(('Invalid SKU Guard', 'Reject unknown SKU', f'PASSED ({e})'))

    # Check 6: Over-transfer / Negative Stock Prevention
    try:
        tracker.stock_sales_floor('WM-2010', 99999)
        results.append(('Over-transfer Guard', 'Block negative backroom stock', 'FAILED'))
    except ValueError as e:
        results.append(('Over-transfer Guard', 'Block negative backroom stock', f'PASSED ({e})'))

    return pd.DataFrame(results, columns=['Audit Check', 'Requirement', 'Result'])

display(run_validation_suite())